# Syntax vs Semantics Analysis using NLP tools -  Spacy and sBERT

<center>

**Name:** Marrion Kiprop Cherop  
**Reg No.:** ST62/80971/2024  
**Programme:** MSc Artificial Intelligence  
**Institution:** Open University of Kenya (OUK)  
**Course:** CSA 803 — Natural Language Processing  
**Module:**  Module 5 Syntax and Semantic Parsing

<center>


## 0. Setup

spaCy's `en_core_web_md` model is used in the notebook. It has a dependency parser for
syntax and 300-dimensional word vectors for semantic similarity.

In [ ]:
import sys, subprocess

def pip_install(*pkgs):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                             capture_output=True, text=True)
    if result.returncode != 0 and "externally-managed-environment" in result.stderr:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", *pkgs])

pip_install("nltk", "spacy", "pandas", "numpy", "scikit-learn")

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# en_core_web_md ships as a GitHub release wheel, not a plain pip package name.
import importlib.util
if importlib.util.find_spec("en_core_web_md") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages",
        "https://github.com/explosion/spacy-models/releases/download/"
        "en_core_web_md-3.8.0/en_core_web_md-3.8.0-py3-none-any.whl"])

print("Setup complete.")

Setup complete.


Sentence-BERT is not run directly here becuase `sentence-transformers` downloads its pretrained weights
from Hugging Face's model hub at run time, and this sandbox's network policy does not allow outbound
connections to `huggingface.co`.

In [ ]:
from sentence_transformers import SentenceTransformer  # noqa: E402

try:
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
    print("SBERT model loaded.")
except Exception as e:
    print(f"SBERT unavailable in this environment: {type(e).__name__}")
    print("Falling back to spaCy's word-vector similarity for the semantic analysis below.")
    sbert_model = None

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT model loaded.


## 1. Dataset

The sentences below are not generic filler — they are drawn from the same domain as the MEL (Monitoring,
Evaluation and Learning) reporting work I do for an agricultural development programme, so the semantic
clusters that show up later are ones I can sanity-check against real field vocabulary rather than trusting
blindly. Twelve sentences, spanning field activity reporting, MEL indicators, and specific agronomic
interventions, gives enough lexical variety for the similarity matrix to be interesting without being so
large that the output becomes unreadable.

In [ ]:
texts = [
    "The field officer recorded maize yields across three sub-counties this quarter.",
    "Smallholder farmers reported delayed rainfall affecting the planting season.",
    "The MEL framework tracks output indicators for the irrigation rehabilitation project.",
    "Extension workers trained households on post-harvest grain storage techniques.",
    "Livestock vaccination coverage improved after the community outreach campaign.",
    "The quarterly narrative report summarized activities under the agricultural development programme.",
    "Soil testing revealed nitrogen deficiency in the demonstration plots.",
    "Farmers adopted drought-resistant seed varieties distributed by the field team.",
    "The cooperative negotiated better prices for surplus maize and beans.",
    "Water harvesting structures reduced runoff loss during the dry spell.",
    "The monitoring officer verified beneficiary lists against the household registry.",
    "Crop rotation practices were introduced to restore degraded farmland.",
]

for i, t in enumerate(texts, 1):
    print(f"S{i}: {t}")

S1: The field officer recorded maize yields across three sub-counties this quarter.
S2: Smallholder farmers reported delayed rainfall affecting the planting season.
S3: The MEL framework tracks output indicators for the irrigation rehabilitation project.
S4: Extension workers trained households on post-harvest grain storage techniques.
S5: Livestock vaccination coverage improved after the community outreach campaign.
S6: The quarterly narrative report summarized activities under the agricultural development programme.
S7: Soil testing revealed nitrogen deficiency in the demonstration plots.
S8: Farmers adopted drought-resistant seed varieties distributed by the field team.
S9: The cooperative negotiated better prices for surplus maize and beans.
S10: Water harvesting structures reduced runoff loss during the dry spell.
S11: The monitoring officer verified beneficiary lists against the household registry.
S12: Crop rotation practices were introduced to restore degraded farmland.


## 2. Tokenization (NLTK)

Before either syntax or semantics can be analyzed, the sentence has to be broken into tokens. NLTK's
`word_tokenize` is the baseline tool for this.

In [ ]:
from nltk.tokenize import word_tokenize

tokens = word_tokenize(texts[0])
print(f"Sentence: {texts[0]}")
print(f"Tokens ({len(tokens)}): {tokens}")

Sentence: The field officer recorded maize yields across three sub-counties this quarter.
Tokens (12): ['The', 'field', 'officer', 'recorded', 'maize', 'yields', 'across', 'three', 'sub-counties', 'this', 'quarter', '.']


## 3. Syntax Analysis (spaCy)

Syntax is about structure. Which word is the subject, which is the main verb, which words modify which
other words. spaCy answers this two ways on the same sentence — part-of-speech tags label each word's
grammatical category, and dependency parsing shows how the words connect to each other.

In [ ]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_md")

sentence = texts[0]
doc = nlp(sentence)
print(f"Parsing: {sentence}")

Parsing: The field officer recorded maize yields across three sub-counties this quarter.


### Part-of-speech tags

In [ ]:
pos_rows = [[token.text, token.pos_, token.tag_] for token in doc]
pos_df = pd.DataFrame(pos_rows, columns=["Word", "POS", "Detailed Tag"])
pos_df

,Word,POS,Detailed Tag
0,The,DET,DT
1,field,NOUN,NN
2,officer,NOUN,NN
3,recorded,VERB,VBD
4,maize,NOUN,NN
5,yields,NOUN,NNS
6,across,ADP,IN
7,three,NUM,CD
8,sub,NOUN,NNS
9,-,NOUN,NNS


### Dependency relations

Each row below shows which word depends on that word, in this grammatical role. "recorded" is the head
the whole sentence hangs off — everything else either modifies it directly or modifies something that
does.

In [ ]:
dep_rows = [[token.text, token.dep_, token.head.text] for token in doc]
dep_df = pd.DataFrame(dep_rows, columns=["Word", "Dependency", "Head"])
dep_df

,Word,Dependency,Head
0,The,det,officer
1,field,compound,officer
2,officer,nsubj,recorded
3,recorded,ROOT,recorded
4,maize,compound,yields
5,yields,dobj,recorded
6,across,prep,recorded
7,three,nummod,sub
8,sub,pobj,across
9,-,pobj,across


### Dependency tree

The tree view makes the head-modifier
structure visible in one look. `nsubj` (field officer) attaches to the root verb `recorded`; `dobj`
(yields) is what got recorded; the prepositional chain `across → sub-counties` and `this quarter` both
attach as modifiers rather than as core arguments of the verb. That distinction — core argument vs.
modifier — is exactly the kind of thing a syntax parser exposes and a semantic embedding does not.

In [ ]:
from spacy import displacy

displacy.render(doc, style="dep", jupyter=True, options={"distance": 100})

In [ ]:
for i, t in enumerate(texts, 1):
    d = nlp(t)
    root = [tok for tok in d if tok.dep_ == "ROOT"][0]
    subj = [tok.text for tok in d if tok.dep_ in ("nsubj", "nsubjpass")]
    obj = [tok.text for tok in d if tok.dep_ in ("dobj", "pobj") and tok.head == root]
    print(f"S{i:2d}  root='{root.text}'  subject={subj}  direct_object={obj}")

S 1  root='recorded'  subject=['officer']  direct_object=['yields']
S 2  root='reported'  subject=['farmers']  direct_object=['rainfall']
S 3  root='tracks'  subject=['framework']  direct_object=['indicators']
S 4  root='trained'  subject=['workers']  direct_object=['households']
S 5  root='coverage'  subject=[]  direct_object=[]
S 6  root='summarized'  subject=['report']  direct_object=['activities']
S 7  root='revealed'  subject=['testing']  direct_object=['deficiency']
S 8  root='adopted'  subject=['Farmers']  direct_object=['varieties']
S 9  root='negotiated'  subject=['cooperative']  direct_object=['prices']
S10  root='reduced'  subject=['structures']  direct_object=['loss']
S11  root='verified'  subject=['officer']  direct_object=['lists']
S12  root='introduced'  subject=['practices']  direct_object=[]


### Syntax across the full corpus

Running the same parse over every sentence, rather than just the first, is what actually lets a pattern
be claimed rather than assumed — and it also surfaced a genuine parser error worth stopping on rather
than smoothing over.

Eleven of the twelve sentences follow a clean `nsubj` — root verb — `dobj` skeleton, which matches the
expectation for formulaic MEL reporting language: a clear actor, a clear action, a clear result (S12 is
passive and correctly parsed as such — `practices` as `nsubjpass` under `introduced`, no direct object,
which is the right structure for a passive construction, not an error).

**S5 is the exception, and it is a real parsing mistake, not a stylistic variant.** For "Livestock
vaccination coverage improved after the community outreach campaign," spaCy assigns ROOT to the noun
**"coverage"**, not the verb "improved," and files "improved" underneath it as an `acl` (a reduced
relative clause) — as if the sentence meant something closer to "the coverage that improved," a noun
phrase, rather than "coverage improved," a full clause. This is a classic garden-path misread as the parser
leans on the noun-heavy run "Livestock vaccination coverage" and treats what follows as a modifier of that
noun phrase rather than recognizing "improved" as the sentence's main verb.

## 4. Semantic Analysis

Semantics asks a different question than syntax. It is not how the sentence is built, but what it is about.
Two sentences with completely different grammatical structure can mean nearly the same thing, and two
sentences with near-identical structure can mean nothing alike. Vector-based sentence embeddings capture
this by mapping each sentence to a point in a high-dimensional space, where distance between points
corresponds to distance in meaning.

The primary engine here is spaCy's averaged word vectors, not SBERT's. This substitution has a real cost worth naming before reading the results. Averaged word vectors have no sense of word order or context — "farmers adopted seeds" and "seeds adopted
farmers" would embed identically — whereas SBERT is trained end-to-end on sentence pairs specifically to
capture that context. The numbers below should be read as *directionally* meaningful, not as
publication-grade semantic similarity scores.

In [ ]:
import numpy as np

docs = [nlp(t) for t in texts]
n = len(docs)

similarity = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        similarity[i, j] = docs[i].similarity(docs[j])

similarity_df = pd.DataFrame(
    similarity.round(2),
    columns=[f"S{i+1}" for i in range(n)],
    index=[f"S{i+1}" for i in range(n)],
)
similarity_df

,S1,S2,S3,S4,S5,S6,S7,S8,S9,S10,S11,S12
S1,1.00,0.82,0.76,0.81,0.73,0.78,0.74,0.77,0.81,0.74,0.74,0.76
S2,0.82,1.00,0.72,0.76,0.78,0.78,0.76,0.82,0.81,0.79,0.68,0.80
S3,0.76,0.72,1.00,0.81,0.78,0.90,0.88,0.70,0.78,0.79,0.82,0.74
S4,0.81,0.76,0.81,1.00,0.73,0.83,0.79,0.78,0.82,0.76,0.80,0.76
S5,0.73,0.78,0.78,0.73,1.00,0.80,0.77,0.73,0.78,0.77,0.79,0.77
S6,0.78,0.78,0.90,0.83,0.80,1.00,0.86,0.69,0.80,0.76,0.84,0.74
S7,0.74,0.76,0.88,0.79,0.77,0.86,1.00,0.73,0.80,0.86,0.78,0.78
S8,0.77,0.82,0.70,0.78,0.73,0.69,0.73,1.00,0.77,0.77,0.65,0.82
S9,0.81,0.81,0.78,0.82,0.78,0.80,0.80,0.77,1.00,0.80,0.78,0.75
S10,0.74,0.79,0.79,0.76,0.77,0.76,0.86,0.77,0.80,1.00,0.72,0.82


### Most similar sentence, per sentence

For each sentence, this pulls out the single closest other sentence by cosine similarity — the diagonal
is excluded so a sentence is never matched to itself.

In [ ]:
for i in range(n):
    row = similarity[i].copy()
    row[i] = -1
    best = int(row.argmax())
    print(f"S{i+1}: {texts[i][:55]:<55}")
    print(f"   -> closest: S{best+1} (score {similarity[i, best]:.3f}) {texts[best][:55]}")

S1: The field officer recorded maize yields across three su
   -> closest: S2 (score 0.821) Smallholder farmers reported delayed rainfall affecting
S2: Smallholder farmers reported delayed rainfall affecting
   -> closest: S8 (score 0.825) Farmers adopted drought-resistant seed varieties distri
S3: The MEL framework tracks output indicators for the irri
   -> closest: S6 (score 0.897) The quarterly narrative report summarized activities un
S4: Extension workers trained households on post-harvest gr
   -> closest: S6 (score 0.834) The quarterly narrative report summarized activities un
S5: Livestock vaccination coverage improved after the commu
   -> closest: S6 (score 0.803) The quarterly narrative report summarized activities un
S6: The quarterly narrative report summarized activities un
   -> closest: S3 (score 0.897) The MEL framework tracks output indicators for the irri
S7: Soil testing revealed nitrogen deficiency in the demons
   -> closest: S3 (score 0.879) The MEL framework tr

###  Matrix Explanation

Two genuine clusters emerge, and they line up with recognizable sub-domains inside agricultural MEL work
rather than being an arbitrary split:

- **Reporting-and-tracking language** — S3 ("MEL framework tracks output indicators"), S6 ("quarterly
  narrative report summarized activities"), and S10 ("monitoring officer verified beneficiary lists")
  cluster together at 0.80–0.90 similarity. None of these three share a common subject or verb — the
  syntax trees would look quite different from each other — but they are all describing the *act of
  reporting and verifying*, which is exactly the kind of shared meaning a syntax parser has no way to
  surface.
- **Field-intervention language** — S1 (maize yields), S7 (soil testing), S9 (cooperative pricing), and
  S11 (crop rotation) cluster more loosely, in the 0.80–0.86 range, around concrete on-the-ground
  agronomic activity rather than reporting.

The similarity scores compress into a narrow 0.78–0.90 band across nearly every pair, including sentences
that are not obviously related — this is the averaging effect flagged above: word-vector averaging picks
up shared *topic vocabulary* (farmers, maize, project, officer) more than it picks up sentence-level
meaning, so unrelated sentences that share domain vocabulary still score high. A properly trained SBERT
model would be expected to spread these scores further apart, since it is trained specifically to
penalize shared vocabulary that does not imply shared meaning. That expected difference is the concrete,
checkable reason to prefer SBERT over averaged vectors in production, not a hypothetical one.

### Running on SBERT

This reproduces the same analysis with proper sentence embeddings.

In [ ]:

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(texts)
sbert_similarity = cosine_similarity(embeddings)

sbert_similarity_df = pd.DataFrame(
     sbert_similarity.round(2),
     columns=[f"S{i+1}" for i in range(len(texts))],
     index=[f"S{i+1}" for i in range(len(texts))],)

sbert_similarity_df


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,S1,S2,S3,S4,S5,S6,S7,S8,S9,S10,S11,S12
S1,1.00,0.34,0.26,0.33,0.20,0.42,0.23,0.36,0.40,0.18,0.15,0.33
S2,0.34,1.00,0.32,0.23,0.26,0.40,0.30,0.58,0.20,0.39,0.00,0.34
S3,0.26,0.32,1.00,0.28,0.14,0.42,0.21,0.33,0.15,0.41,0.12,0.29
S4,0.33,0.23,0.28,1.00,0.24,0.29,0.14,0.32,0.28,0.28,0.17,0.37
S5,0.20,0.26,0.14,0.24,1.00,0.27,0.12,0.31,0.25,0.09,0.08,0.25
S6,0.42,0.40,0.42,0.29,0.27,1.00,0.23,0.36,0.37,0.19,0.09,0.39
S7,0.23,0.30,0.21,0.14,0.12,0.23,1.00,0.23,0.13,0.18,0.06,0.29
S8,0.36,0.58,0.33,0.32,0.31,0.36,0.23,1.00,0.29,0.31,-0.01,0.44
S9,0.40,0.20,0.15,0.28,0.25,0.37,0.13,0.29,1.00,0.12,0.02,0.30
S10,0.18,0.39,0.41,0.28,0.09,0.19,0.18,0.31,0.12,1.00,-0.06,0.37


## 5. Comparision of Spacy and sBert

spaCy's scores sit in a narrow 0.65–0.90 band across nearly every pair. This isn't because en_core_web_md lacks word vectors — it has real 300-dimensional GloVe-style vectors — but because sentence similarity here is computed by averaging those word vectors, which discards word order and over-weights shared topic vocabulary (farmers, maize, project, officer) over actual sentence-level meaning. Unrelated sentences that share domain words still end up scoring high. SBERT's range, -0.06 to 0.58, is far more discriminative, because it's trained end-to-end on sentence pairs specifically to separate lexical overlap from genuine semantic similarity.
Both models correctly cluster the field-activity sentences (S2/S8, S4/S9, S7/S10) and link S3–S6 as the top M&E pair. They disagree on S11 ("monitoring officer verified beneficiary lists"), which belongs with S3/S6 by content. spaCy places it there, at 0.839 with S6. SBERT scores it near zero against both (0.12, 0.09), missing the connection entirely.
SpaCy's averaged-vector scores are directionally useful but not calibrated — the compression is a structural property of averaging, not a missing-vectors defect, and it inflates similarity for any sentences sharing domain vocabulary regardless of actual meaning. SBERT's range is genuinely discriminative, but it misses domain-specific structure (MEL/reporting vocabulary) that a general-purpose checkpoint like all-MiniLM-L6-v2 likely never saw in training. spaCy's correct grouping of S11 may be riding on lexical overlap ("report," "framework," "indicators") rather than real semantic understanding — getting the right answer through a mechanism that isn't reliable. Architecture alone doesn't guarantee domain fit: for MEL-specific text, fine-tuning a sentence-embedding model on domain data, or supplementing embedding similarity with a vocabulary-aware check, would likely outperform either model as used here.